# 45. Orders B and C, five seeds each, with the indicators attached

This is job 3 of three in the revised plasticity study. It is the shorter of
the two training jobs.

## What this notebook does

It trains the same nine tasks twice more, in two different orders, with the
same seed and the same learning rate as job 2. Then it unlearns the first task
of each order.

Three orders exist in the study:

| Order | Task sequence | Position 1 / 5 / 9 |
|---|---|---|
| order_A | 0, 1, 2, 3, 4, 5, 6, 7, 8 | 0 / 4 / 8 |
| order_B | 4, 1, 2, 3, 8, 5, 6, 7, 0 | 4 / 8 / 0 |
| order_C | 8, 1, 2, 3, 0, 5, 6, 7, 4 | 8 / 0 / 4 |

order_A was job 2. This notebook runs B and C.

## Why the orders are built this way

Six of the nine tasks sit in the same position in all three orders. Only tasks
0, 4 and 8 move, and they rotate: each one is learned first in one order, in
the middle in another, and last in the third.

That rotation is the whole design. If you shuffled all nine tasks and found
that one order read differently from another, you could not say whether the
order mattered or whether the tasks that happened to come first were simply
harder. Here the same three tasks visit the same three positions, so the task
and the position come apart.

## What it is evidence for

**Section 4.2, task order.** The claim is that changing only the order changes
the reported fall in effective rank by a large factor, while the generator
travels the same distance.

Five seeds per ordering, matching job 2, so a gap between orders is read
against how far one order's own seeds disagree. One seed per extra
order means the direction of the effect is supported and the size of it is one
measurement per order. The paper must say so.

**Section 4.3, weight scale.** Two more pairs of saved models, which brings
the total to seven across three orders and five seeds.

## Before you run this

Job 2 must have passed its gate. If the five seeds did not land within 3
percentage points of each other, the setting is not good enough to measure
against and these two runs would be measured on top of noise.

Use the same `LEARNING_RATE` job 2 used. A different value here would mean the
order comparison also changes the learning rate, and neither effect could be
read on its own.

## Where to run it

Either machine works, and this one is short enough that Colab is the easier
choice: two nine-task runs at about 1.4 hours each on an L4, so under 3 hours.

If the session does drop, start the notebook again from the top. Everything is
in R2, an order that already finished is skipped without training, and a
half-finished one picks up from its last boundary.

## 1. Load the code

In [ ]:
from pathlib import Path
import sys, os, json, time, shutil, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules

# Importing uncle twice in one session gives two copies of the module with two
# copies of its constants, which is a confusing way to fail much later.
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")

# This notebook runs at the current main. The nine-task design and the
# ten-group class split were added after the earlier runs, so a pinned
# checkout would not contain them.
if IN_COLAB:
    REPO = Path("/content/path-plasticity")
    if not REPO.exists():
        subprocess.run(["git", "clone",
                        "https://github.com/sumitasthana/PATH.git", str(REPO)],
                       check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    # On RunPod, clone the repository yourself and start Jupyter inside it.
    # This walks up from the working directory to find the checkout.
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")

for package in ("boto3", "tqdm"):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                        package], check=True)
sys.path.insert(0, str(REPO))

import torch
from uncle import Config, run
from uncle.config import study_defaults, PROBE_TASK
from uncle.storage import R2Store
from uncle.streams import build_tasks
from uncle.tasks import PARTITIONS
from uncle.plasticity_indicators import (
    StreamObserver, fixed_image_sets, indicator_key)

HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                               text=True).strip()
print("Running at:", HEAD)
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)
# Let the tensor cores do the matrix multiplies in TF32. TF32 keeps the range
# of a normal 32-bit float but drops some of the digits after the point. The
# loss is far smaller than the difference between two random seeds, and both
# the A100 and the L4 run several times faster this way. If no A100 is free
# and the run lands on an L4, this is the single setting that matters most.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
# Every image here is 64 by 64 and never changes size, so letting cuDNN spend
# the first few batches picking its fastest algorithm pays for itself.
torch.backends.cudnn.benchmark = True

if torch.cuda.is_available():
    memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU:", torch.cuda.get_device_name(0), f"| {memory:.0f} GB")
    # Where the memory goes at the ninth task, which is the heaviest point:
    # the generator and its Adam state are about 2.1 GB, the eight protected
    # tasks hold one frozen weight set each at 94 MB, and the protection term
    # builds eight more with their gradients. The peak measured this way is
    # roughly 8 GB, so a 24 GB L4 fits. An L4 is slower than an A100, not
    # smaller: expect two to three times the wall clock, not an out-of-memory
    # error. The first run prints a projection so you can decide then.
    if memory < 20:
        print("Under 20 GB. The peak here is near 8 GB at the ninth task, so "
              "this should still fit. Watch the first run before trusting it.")
else:
    print("No GPU found. This notebook trains for hours. Do not start it on CPU.")

## 2. Settings

In [ ]:
# The same learning rate job 2 used. If this differs from job 2, the order
# comparison changes two things at once and neither can be read on its own.
LEARNING_RATE = 5e-4          # the same value notebook 44 used

if LEARNING_RATE is None:
    raise ValueError(
        "Set LEARNING_RATE to the value job 2 used. It is in "
        "job2_summary.json under 'learning_rate'.")

# Hand the GPU back when the last cell confirms every file is in R2. A
# Colab GPU keeps billing while it is attached, and these runs finish hours
# after anyone is watching. Set this to False to keep the runtime for
# inspection after the run.
RELEASE_GPU_WHEN_DONE = True

STUDY_ID = "plasticity_revised_v1"
BUCKET = "uncle-experiments"

# The two further orderings, five seeds each.
#
# One seed each was the original plan. Reading job 2's five records killed it.
# Effective rank moves about 1.33 between seeds of the same order, while nine
# tasks of learning move it about 2.78. So a gap between two single runs has
# to clear roughly 2.0, which is 72 percent of the measure's own value, before
# it means anything. srank is worse. The weight measures would have been fine
# on one seed, but the rank measures are the ones section 4.2 leads with.
#
# Five seeds per order turns that comparison from "one number against one
# number" into "one average against another, each with a spread". Ten runs at
# about 0.7 hours each.
#
# The seeds match job 2's, so order_A, order_B and order_C can be compared
# seed for seed as well as average to average.
#
# The task unlearned is the one learned first in that ordering, matching what
# job 2 did for order_A.
ORDERS = {
    "order_B": {"forget": "4"},
    "order_C": {"forget": "8"},
}
SEEDS = (0, 1, 2, 3, 4)

# Must match job 2 exactly. These are not knobs here.
EPOCHS = 5
BATCH_SIZE = 64
INITIALIZATION = "hyperfan_in"

IMAGES_PER_SET = 256
INCLUDE_EUROSAT = True

# Same reasoning as job 2: boundary 9 is the end of the stream, boundary 10 is
# after the unlearn, boundary 4 is a mid-stream reference. A checkpoint holds
# the whole 134-million-parameter generator, so each file is about 550 MB:
# roughly 1.7 GB per run. At ten runs that is about 17 GB for this notebook,
# so check the size of the first upload before trusting that figure. Drop this
# to (9, 10) to halve it; boundary 4 is only a convenience.
KEEP_BOUNDARIES = (4, 9, 10)

DATA_ROOT = (Path("/content/data/tiny-imagenet-200") if IN_COLAB
             else REPO / "data/tiny-imagenet-200")
EUROSAT_ROOT = Path("/content/eurosat") if IN_COLAB else REPO / "data/eurosat"
OUTPUT_ROOT = Path("/content/job3") if IN_COLAB else REPO / "outputs/job3-orders"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Nine learns, then one forget. One entry per run, so an order with five seeds
# is five entries. The key is the run's name and its folder.
PLAN = {}
for order, choice in ORDERS.items():
    stream = study_defaults(order)["requests"]
    for seed in SEEDS:
        PLAN[f"{order}_seed{seed}"] = {
            "order": order,
            "seed": seed,
            "forget": choice["forget"],
            "stream": stream,
            "requests": stream + (("forget", choice["forget"]),),
        }

# Every task any run here needs. The nine are the same in both orders.
ALL_REQUESTS = PLAN[f"order_B_seed{SEEDS[0]}"]["requests"]

print("Study:", STUDY_ID)
for order, choice in ORDERS.items():
    stream = study_defaults(order)["requests"]
    print(f"  {order}  seeds {', '.join(str(s) for s in SEEDS)}  "
          f"stream {' '.join(task for _, task in stream)}  "
          f"then unlearn {choice['forget']}")
print("Held out as the probe group, never trained:", PROBE_TASK)
print("Learning rate:", LEARNING_RATE, "| epochs:", EPOCHS,
      "| batch:", BATCH_SIZE)
print("Runs in this job:", len(PLAN),
      "| task-trainings:", 9 * len(PLAN),
      "| rough hours at 0.7 per run:", round(0.7 * len(PLAN), 1))

## 3. Connect to R2

R2 is the cloud bucket. It is what makes the split between RunPod and Colab
safe: this notebook writes its records and checkpoints there as it goes, and
the analysis notebooks read the same files from Colab afterwards.

The rule is simple. If a result is not in R2, it does not exist. A RunPod
instance that dies holding nine tasks of unsaved work costs real money to
repeat.

You need three secrets: `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and
`R2_SECRET_ACCESS_KEY`. On Colab put them in the secrets panel on the left.
On RunPod set them as environment variables before starting Jupyter.

In [ ]:
def secret(name):
    """Read a credential from the environment, or from Colab's secret store."""
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
                        secret("R2_ACCESS_KEY_ID"),
                        secret("R2_SECRET_ACCESS_KEY"))
store.progress = True

CACHE = OUTPUT_ROOT / "cache"
CACHE.mkdir(parents=True, exist_ok=True)

print("Connected to R2 bucket:", BUCKET)
print("Records go under: uncle/indicators/" + STUDY_ID + "/")

## 4. Build the image sets

This is the slow part before training starts. Indexing the dataset costs more
than the first optimizer steps do, so it is done once here and every run in
this notebook reuses it.

Three sets of pictures are measured at every boundary, and they answer
different questions.

- `in_stream` is held-out test images of the nine tasks the model learns. It
  says what happens to data the model is being trained on.
- `probe` is the tenth group of classes, which is never trained on. Same
  dataset, same kind of picture, classes the model has never seen.
- `eurosat` is satellite photographs. Nothing about them resembles the
  training stream.

A measure that moves on all three is describing the generator. A measure that
moves only on `in_stream` is describing what the model learned about those
particular classes.

The same 256 pictures are used at every boundary and in every run. If the
pictures moved between two measurements, a change in the reading could mean
the model changed or the pictures changed, and there would be no way to tell
which. The adapter plasticity study made exactly that mistake and had to redo
its analysis.

In [ ]:
from torch.utils.data import ConcatDataset

# Every task any run in this notebook needs, built in one call. The nine
# learned groups are the same in every ordering, only their order differs.
STREAM_TASKS = sorted({task for _, task in ALL_REQUESTS}, key=int)

tasks = build_tasks(root=DATA_ROOT,
                    partition_path=PARTITIONS[20],
                    partition_seed=42,
                    include=tuple(STREAM_TASKS + [PROBE_TASK]),
                    download=True)
print("Loaded Tiny ImageNet images for", len(tasks), "tasks")

sources = {
    "in_stream": ConcatDataset([tasks[name]["test"] for name in STREAM_TASKS]),
    "probe": tasks[PROBE_TASK]["test"],
}

if INCLUDE_EUROSAT:
    from torchvision.datasets import EuroSAT
    from torchvision.transforms import ToTensor
    # Already 64 by 64, which is the size the backbone expects.
    eurosat = EuroSAT(root=str(EUROSAT_ROOT), transform=ToTensor(),
                      download=True)
    assert eurosat[0][0].shape == (3, 64, 64), (
        f"Unexpected shape {eurosat[0][0].shape}")
    sources["eurosat"] = eurosat

# The draw is seeded inside the library, not here, so every run of every
# notebook in this study reads the same pictures.
image_sets = fixed_image_sets(sources, count=IMAGES_PER_SET)
for name, images in image_sets.items():
    print(f"{name:<10} {tuple(images.shape)}")

## 5. Build the two configurations

One settings object per ordering. Everything except the request order and the
unlearned task is identical, and identical to job 2.

The cell also checks that the three orderings really do form the rotation the
design claims. If somebody edits the orderings later and breaks the pattern,
this stops the run rather than producing a comparison that cannot be read.

In [ ]:
configs = {}
for label, step in PLAN.items():
    settings = study_defaults(step["order"])
    settings["requests"] = step["requests"]
    configs[label] = Config(
        **settings,
        seed=step["seed"],
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        initialization=INITIALIZATION,
        device="cuda" if torch.cuda.is_available() else "cpu",
    )

# The rotation check. Positions 1, 5 and 9 are indices 0, 4 and 8 of the
# stream. Across the three orderings each moving task must appear once in
# each of those three positions, and the other six must never move.
rotating = {}
fixed = {}
for order in ("order_A", "order_B", "order_C"):
    stream = [task for _, task in study_defaults(order)["requests"]]
    for position, task in enumerate(stream):
        if position in (0, 4, 8):
            rotating.setdefault(task, set()).add(position)
        else:
            fixed.setdefault(position, set()).add(task)

if sorted(rotating) != ["0", "4", "8"]:
    raise ValueError(f"Expected tasks 0, 4 and 8 to rotate, got {sorted(rotating)}")
for task, positions in rotating.items():
    if positions != {0, 4, 8}:
        raise ValueError(f"Task {task} does not visit all three positions: {positions}")
for position, seen in fixed.items():
    if len(seen) != 1:
        raise ValueError(f"Position {position} is not fixed: {seen}")
print("Rotation check passed: tasks 0, 4 and 8 each visit positions 1, 5 and 9.")

example = configs[next(iter(PLAN))]
CHANCE = 100 / example.classes_per_task
print()
print("classes per task:", example.classes_per_task,
      "| chance accuracy:", f"{CHANCE:.0f}%")
print("backbone:", example.backbone, "| beta:", example.beta,
      "| burn-in steps for the unlearn:", example.burn_in)
# One line per order, not per run: the requests are the same for all five
# seeds of an order, only the random start differs.
for order in ORDERS:
    label = f"{order}_seed{SEEDS[0]}"
    print(order, "requests:",
          " ".join(f"{action[0].upper()}{task}"
                   for action, task in configs[label].requests))

## 6. Run the streams

Each run does the nine tasks, then the unlearn, and measures the indicators at
every boundary along the way.

Three things make an interrupted session cheap to recover.

The training state is saved after every request, and the run picks it up on a
second call. Re-running this cell after a disconnect continues at the task it
died on, not at the top.

The indicator record is uploaded to R2 after every boundary, not at the end. A
boundary already in the record is skipped rather than measured again.

The boundary models named in `KEEP_BOUNDARIES` are uploaded as they are
written, not at the end of the run.

**One limitation, stated plainly.** The within-task loss curve that notebook
36 recorded is not collected here. The library calls the observer at request
boundaries but has no per-update hook on this code path, so those rows would
be empty. Every boundary indicator the paper uses is collected. Only the
curve inside a task is missing, and nothing in the paper's claims rests on it.

In [ ]:
from datetime import datetime
from statistics import mean


def clock():
    """The wall clock, so a printed line can be placed in the day."""
    return datetime.now().strftime("%H:%M:%S")


def spell(seconds):
    """A duration as h:mm:ss. These run to hours, so minutes alone would not do."""
    seconds = int(max(seconds, 0))
    return f"{seconds // 3600}:{seconds // 60 % 60:02d}:{seconds % 60:02d}"


try:
    from tqdm.auto import tqdm as _tqdm
except ImportError:
    _tqdm = None


def say(text=""):
    """Print a line without breaking the progress bar.

    The trainer draws a tqdm bar while it works. An ordinary print() during
    that writes over the bar's line, so the bar's leftover characters stay on
    screen and the log fills with ragged part-lines and trailing blanks.
    tqdm.write() erases the bar first, prints, then draws the bar again below.
    Where tqdm is not installed there is no bar to protect, so print is fine.
    """
    if _tqdm is None:
        print(text, flush=True)
    else:
        _tqdm.write(text)


def run_one(label, config, order, seed, forget_task, requests):
    """Train one stream, unlearn one task, measure every boundary.

    `label` names the run and its output folder. `requests` is the nine learns
    followed by the one forget, so the last boundary is the damaged model.
    Returns a summary dictionary; the full record is the JSONL file in R2.
    """
    output = OUTPUT_ROOT / label
    output.mkdir(parents=True, exist_ok=True)
    run_started = time.perf_counter()
    total_boundaries = len(requests)

    log_key = indicator_key(STUDY_ID, order, seed)
    baseline_key = log_key.replace(".jsonl", ".baseline.pt")
    log_path = output / "indicators.jsonl"
    baseline_path = log_path.with_suffix(".baseline.pt")
    checkpoint_path = output / "checkpoint.pt"

    # The training checkpoint also lives in R2, under one key that is
    # overwritten after every request. The machine's own disk is not a safe
    # place for it: a RunPod pod that goes away takes its disk with it, and a
    # Colab runtime recycles. Without this copy a restarted run would find its
    # indicator record waiting in R2 and then retrain nine tasks from zero to
    # refill a record it already has.
    resume_key = (f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}"
                  f"/resume.pt")

    # An interrupted run picks up its own record. Both files come back or
    # neither: relative weight change is measured against boundary zero, and
    # the record without its baseline would have nothing to measure against.
    if store.info(log_key) is not None:
        if store.info(baseline_key) is None:
            raise RuntimeError(
                f"{log_key} is in R2 but {baseline_key} is missing. Boundary "
                "zero cannot be recreated once the generator has moved. "
                "Delete the record and start this run again.")
        shutil.copyfile(store.download(log_key, CACHE), log_path)
        shutil.copyfile(store.download(baseline_key, CACHE), baseline_path)
        done = [json.loads(line) for line
                in log_path.read_text(encoding="utf-8").splitlines()
                if line.strip()]
        say(f"  resuming: {len(done)} boundaries already measured, "
            f"up to t = {max(row['t'] for row in done)}")
    else:
        log_path.unlink(missing_ok=True)
        baseline_path.unlink(missing_ok=True)
        say("  starting fresh: no record in R2 yet")

    # Bring the training checkpoint back before the run starts. Only when
    # there is no local copy: a local file is the newer of the two, because it
    # is written first and uploaded after.
    if not checkpoint_path.exists() and store.info(resume_key) is not None:
        shutil.copyfile(store.download(resume_key, CACHE), checkpoint_path)
        say("  training checkpoint restored from R2")

    observer = StreamObserver(log_path, image_sets,
                              study=STUDY_ID, order=order, seed=seed)
    handle = {}

    def upload_record():
        """Push the record and its baseline to R2, replacing the last copy."""
        if log_path.exists():
            store.upload(log_path, log_key, overwrite=True)
        if baseline_path.exists():
            store.upload(baseline_path, baseline_key, overwrite=True)

    def remember(info):
        """Called before the first request, with the trainer the run built.

        Boundary zero is measured here: the fresh generator, before anything
        is learned. It is the value every later weight change is measured
        against, which is why it cannot be skipped. On a resume this is called
        a second time with `resumed_after`, and boundary zero is already in
        the record, so the observer returns from it without measuring again.
        """
        handle["uncle"] = info["uncle"]
        observer.boundary(info["uncle"], 0, None)
        upload_record()

    def measure(record):
        """Called after every finished request. Boundary t is index + 1."""
        t = record["index"] + 1
        row = observer.boundary(handle["uncle"], t, record["task"])
        upload_record()
        did = "learn" if record["action"] == "learn" else "UNLEARN"
        accuracy = record["after"][record["task"]]

        # Every line has the same columns in the same places, so the log reads
        # down as a table and a changing number is easy to spot. A measure the
        # observer did not return prints as a dash rather than vanishing,
        # because a missing column would shift everything after it.
        def column(value, spec):
            return format(value, spec) if value is not None else "-".rjust(
                len(format(0, spec)))

        row = row or {}
        line = (
            f"  {clock()}  {label:<14} "
            f"t {t:>2}/{total_boundaries}  "
            f"{did:<7} task {record['task']:<2} "
            f"acc {accuracy:5.1f}%  "
            f"rank {column(row.get('effective_rank_in_stream'), '6.2f')}  "
            f"norm {column(row.get('norm_generator'), '7.1f')}  "
            f"moved {column(row.get('delta_w_generator'), '6.4f')}  "
            f"{spell(time.perf_counter() - run_started)}"
        )
        say(line)

    def keep_model(record, saved_path):
        """Send the saved checkpoint to R2 after every finished request.

        Two copies, for two different jobs. The resume copy is overwritten
        every time and is what a restarted run picks up. The boundary copies
        are separate files that stay, because the rescaling control needs the
        model from before the unlearn and the model from after it, and the
        resume copy will have moved past both by then.
        """
        t = record["index"] + 1
        saved_path = Path(saved_path)
        try:
            store.upload(saved_path, resume_key, overwrite=True)
        except Exception as failure:
            # A storage hiccup is not worth losing an hour of GPU time over.
            # The record for this boundary is already uploaded, and the next
            # request tries this upload again.
            say(f"        could not save the resume point: {failure}")
        if t not in KEEP_BOUNDARIES:
            return
        key = (f"uncle/checkpoints/{STUDY_ID}/{order}/seed_{seed}"
               f"/boundary_{t}.pt")
        try:
            store.upload(saved_path, key, overwrite=True)
            say(f"        kept the boundary {t} model in R2")
        except Exception as failure:
            say(f"        could not keep the boundary {t} model: {failure}")

    began = time.perf_counter()
    history = run(
        config,
        tasks=tasks,
        on_start=remember,
        on_request=measure,
        progress=True,
        checkpoint_path=checkpoint_path,
        resume=True,
        on_saved_checkpoint=keep_model,
    )
    took = time.perf_counter() - began

    # The record before the unlearn is the stream's own result. The last
    # record is after the unlearn, where one task is destroyed on purpose, so
    # reading mean accuracy there would mix damage into the gate.
    learned = [task for action, task in requests if action == "learn"]
    end_of_stream = history[len(learned) - 1]["after"]
    per_task = {task: end_of_stream[task] for task in learned}

    return {
        "run": label, "order": order, "seed": seed,
        "learning_rate": LEARNING_RATE,
        "accuracy_per_task": per_task,
        "mean_accuracy": mean(per_task.values()),
        "forgot_task": forget_task,
        "accuracy_of_forgotten_task": history[-1]["after"][forget_task],
        "boundaries_measured": len(requests) + 1,
        "wall_clock_seconds": took,
        "code_revision": HEAD,
    }

In [ ]:
results = {}

for index, (label, step) in enumerate(PLAN.items(), start=1):
    say()
    say("=" * 96)
    say(f"[{index}/{len(PLAN)}] {label}  "
        f"unlearn task {step['forget']} at the end  |  started {clock()}")
    say(f"    columns: time | run | boundary | action | accuracy of that task "
        f"| effective rank | generator norm | distance moved | elapsed")
    say("=" * 96)

    row = run_one(label=label, config=configs[label],
                  order=step["order"], seed=step["seed"],
                  forget_task=step["forget"], requests=step["requests"])
    results[label] = row

    say()
    say(f"  mean accuracy over the nine tasks: {row['mean_accuracy']:.2f}%")
    say(f"  task {step['forget']} after the unlearn: "
        f"{row['accuracy_of_forgotten_task']:.2f}%  "
        f"(chance is {CHANCE:.0f}%)")
    say(f"  wall clock {spell(row['wall_clock_seconds'])}")

## 7. The result

The comparison this job exists for is not in this table. It is in the
indicator records: how far effective rank falls from boundary 0 to boundary 9
in each of the three orders, against how far the generator travelled. That
comparison is made in the analysis notebook, on Colab, from the records in R2.

What this table does show is whether the three orders reached comparable
accuracy. They should. If one order finished far behind the others, the later
rank comparison is reading two different models and not two different orders.

Each order now has five seeds, so the table gives an average and a spread for
each one rather than a single number. The gap between order averages is only a
difference if it is wider than the seeds of one order disagree among
themselves.

In [ ]:
from statistics import mean as avg, stdev

print(f"{'run':>18}{'mean accuracy':>16}{'unlearned task after':>22}{'hours':>8}")
for label in PLAN:
    if label not in results:
        print(f"{label:>18}{'not run':>16}")
        continue
    row = results[label]
    print(f"{label:>18}{row['mean_accuracy']:>15.2f}%"
          f"{row['accuracy_of_forgotten_task']:>21.2f}%"
          f"{row['wall_clock_seconds'] / 3600:>8.1f}")

# Per order: the average over its seeds, and how far the seeds disagree. This
# pair of numbers is the whole reason for running five seeds instead of one.
by_order = {}
for label, step in PLAN.items():
    if label in results:
        by_order.setdefault(step["order"], []).append(
            results[label]["mean_accuracy"])

print()
print(f"{'order':>10}{'seeds':>7}{'average':>10}{'spread':>9}{'sd':>8}")
for order, scores in by_order.items():
    sd = stdev(scores) if len(scores) > 1 else float("nan")
    print(f"{order:>10}{len(scores):>7}{avg(scores):>9.2f}%"
          f"{max(scores) - min(scores):>8.2f}{sd:>8.2f}")

# Job 2's numbers, if its summary is in R2, so order_A joins the comparison
# here instead of by hand later.
job2_scores = None
job2_key = f"uncle/indicators/{STUDY_ID}/job2_summary.json"
if store.info(job2_key) is not None:
    job2 = json.loads(store.download(job2_key, CACHE).read_text(encoding="utf-8"))
    job2_scores = [row["mean_accuracy"] for row in job2["results"]]
    sd = stdev(job2_scores) if len(job2_scores) > 1 else float("nan")
    print(f"{'order_A':>10}{len(job2_scores):>7}{avg(job2_scores):>9.2f}%"
          f"{max(job2_scores) - min(job2_scores):>8.2f}{sd:>8.2f}")

# Does the order change accuracy at all? Compare the gap between order
# averages against how much one order's own seeds disagree. A gap smaller than
# the seed spread is not a difference between orders.
averages = {order: avg(scores) for order, scores in by_order.items()}
if job2_scores:
    averages["order_A"] = avg(job2_scores)
if len(averages) > 1:
    gap = max(averages.values()) - min(averages.values())
    within = [stdev(s) for s in by_order.values() if len(s) > 1]
    if job2_scores and len(job2_scores) > 1:
        within.append(stdev(job2_scores))
    noise = avg(within) if within else float("nan")
    print()
    print(f"Widest gap between order averages: {gap:.2f} points.")
    print(f"Typical disagreement between seeds of one order: {noise:.2f} points.")
    if gap > 2 * noise:
        print("The orders differ by more than the seeds do.")
    else:
        print("The gap between orders is inside the seed noise, so these "
              "orders reached comparable accuracy.")
print()
print("Accuracy is not the comparison this job exists for. That one is the "
      "rank measures, read from the records in R2 by the analysis notebook.")

summary = {
    "study": STUDY_ID, "job": 3, "code_revision": HEAD,
    "orders": {order: list(SEEDS) for order in ORDERS},
    "learning_rate": LEARNING_RATE,
    "classes_per_task": example.classes_per_task,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "beta": example.beta, "backbone": example.backbone,
    "initialization": INITIALIZATION,
    "kept_boundaries": list(KEEP_BOUNDARIES),
    "results": [results[label] for label in PLAN if label in results],
    "order_averages": averages,
    "job2_scores": job2_scores,
}
summary_path = OUTPUT_ROOT / "job3_summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
store.upload(summary_path, f"uncle/indicators/{STUDY_ID}/job3_summary.json",
             overwrite=True)
print()
print("Written:", summary_path)

# Read the object back from R2 rather than trusting the upload call. The
# release cell below refuses to hand the GPU back unless this is True, so a
# failed save can never cost the whole run.
SAVE_VERIFIED = (len(results) == len(PLAN) and store.info(
    f"uncle/indicators/{STUDY_ID}/job3_summary.json") is not None)
print("Summary confirmed in R2." if SAVE_VERIFIED
      else "Summary NOT confirmed in R2. Do not close this runtime.")

## 8. What comes next

All seven stream runs are done: five seeds of order_A, one each of order_B and
order_C, each ending in a deliberate unlearn. That is every training run the
paper needs.

Everything left after job 3 reads files and trains nothing, so it runs on
Colab:

- the rescaling control, four forward passes per pair of saved models
- the tables and figures, from the indicator records in R2

Nothing from this notebook has to be recomputed on another machine. The
records and the kept boundary models are in R2, and the later notebooks read
them from there.

## 9. Hand the GPU back

Releases the Colab runtime once the work is done. It refuses to release if a run is
missing or the summary is not confirmed in R2, so a failed save cannot cost you the run.


In [ ]:
if not SAVE_VERIFIED:
    print("Not releasing the GPU: a run is missing or the summary is not in R2.")
    print("Rerun section 7 before closing this runtime, or the results are lost.")
elif not RELEASE_GPU_WHEN_DONE:
    print("Results are in R2 under:",
          f"uncle/indicators/{STUDY_ID}/")
    print("RELEASE_GPU_WHEN_DONE is off, so the runtime stays connected.")
elif not IN_COLAB:
    # On RunPod there is no runtime to unassign. Free the memory anyway so
    # anything else on the machine can use the card.
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Not running in Colab, so there is no runtime to release.")
    print("Stop the pod yourself when you are done with it.")
else:
    print("Results are in R2 under:", f"uncle/indicators/{STUDY_ID}/")
    import gc
    # Drop the big objects first. Without this the generator and its Adam
    # state stay referenced and empty_cache has nothing to give back.
    for name in ("results", "configs", "tasks", "observer"):
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    from google.colab import runtime
    print("Releasing the Colab runtime now.")
    runtime.unassign()
